# LACK Table — Predicate & Action Invention via Physics Simulation

**Goal.** Given (a) an IKEA manual step illustration and (b) 3D CAD meshes of the parts,
use physics simulation to *imagine* boolean predicates and *induce* a STRIPS-style action schema.

**Approach.**
1. Load 6 LACK part meshes + the step-0 manual illustration from this repo.
2. Build a MuJoCo 3.x scene with free-body parts.
3. Sample random initial configurations (uniform xy × Shoemake-uniform SO(3)).
4. Apply a primitive action `drop(i, x, y)` to collect (before, after) transitions.
5. Send (manual, before-render, after-render) to a VLM → get candidate predicates as Python expressions.
6. Sim-verify the predicates, then induce a (pre, add, del) schema for the action.

**Runtime.** Google Colab, T4 or A100 (A100 is smoother for the VLM cell). Run cells top-to-bottom.

In [4]:
# --- Setup ----------------------------------------------------------------
# Install deps + clone this repo so the Colab kernel can see the LACK meshes
# and the manual illustrations shipped under VLM_assembly_plan_gen/data/.
import os, sys, subprocess, shutil
from pathlib import Path

REPO_URL = "https://github.com/Praveen1098/IKEA-Assembly-Planning.git"
ON_COLAB = Path("/content").exists()
REPO_DIR = Path("/content/IKEA-Assembly-Planning") if ON_COLAB else Path.cwd()

def sh(cmd, **kw):
    print("$", " ".join(cmd)); subprocess.check_call(cmd, **kw)

sh([sys.executable, "-m", "pip", "install", "-q",
    "mujoco>=3.2", "trimesh>=4.0", "lxml>=5.0",
    "opencv-python-headless>=4.9", "Pillow", "scipy>=1.11",
    "transformers>=4.49", "accelerate>=0.33", "bitsandbytes>=0.43"])

if ON_COLAB and not REPO_DIR.exists():
    sh(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)])

os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
os.environ.setdefault("MUJOCO_GL", "egl")  # headless GL for MuJoCo on Colab

print("repo dir:", REPO_DIR)
print("cwd     :", Path.cwd())

$ /usr/bin/python3 -m pip install -q mujoco>=3.2 trimesh>=4.0 lxml>=5.0 opencv-python-headless>=4.9 Pillow scipy>=1.11 transformers>=4.49 accelerate>=0.33 bitsandbytes>=0.43
$ git clone --depth 1 https://github.com/Praveen1098/IKEA-Assembly-Planning.git /content/IKEA-Assembly-Planning
repo dir: /content/IKEA-Assembly-Planning
cwd     : /content/IKEA-Assembly-Planning


## 1. Load LACK parts and the manual step illustration

LACK has 6 OBJ meshes (`00.obj`…`05.obj`). By inspection of the meshes:
part `0` is the tabletop slab, parts `2-5` are the four legs. Step 0 of the manual shows
the four legs being attached to the underside of the tabletop.

In [5]:
import numpy as np, trimesh, matplotlib.pyplot as plt
from PIL import Image

DATA      = REPO_DIR / "VLM_assembly_plan_gen" / "data"
PARTS_DIR = DATA / "parts" / "Table" / "lack"
MASK_DIR  = DATA / "mask"  / "Table" / "lack"

PART_IDS = [0, 1, 2, 3, 4, 5]
MESH = {i: trimesh.load(str(PARTS_DIR / f"{i:02d}.obj"), force="mesh") for i in PART_IDS}
for i, m in MESH.items():
    print(f"part {i}: verts={len(m.vertices):4d}  extents={m.extents.round(3)}  centroid={m.centroid.round(3)}")

manual_img = Image.open(MASK_DIR / "step_0_no_seg.png").convert("RGB")
plt.figure(figsize=(6, 4)); plt.imshow(manual_img); plt.axis("off")
plt.title("LACK manual — step 0"); plt.show()

ValueError: string is not a file: `/content/IKEA-Assembly-Planning/VLM_assembly_plan_gen/data/parts/Table/lack/00.obj`

## 2. Build a MuJoCo scene from the OBJ meshes

One free body per part with a mesh geom. MuJoCo computes convex-hull collision geometry
automatically. Floor + overhead camera + one light.

In [ ]:
import mujoco

SCENE_DIR   = Path("/tmp/lack_scene"); SCENE_DIR.mkdir(exist_ok=True)
MESH_SUBDIR = SCENE_DIR / "meshes";    MESH_SUBDIR.mkdir(exist_ok=True)
for i in PART_IDS:
    shutil.copy(str(PARTS_DIR / f"{i:02d}.obj"), str(MESH_SUBDIR / f"{i:02d}.obj"))

def build_scene_xml():
    meshes = "\n".join(f'    <mesh name="m{i}" file="{i:02d}.obj"/>' for i in PART_IDS)
    bodies = []
    for i in PART_IDS:
        bodies.append(
            f'  <body name="part_{i}" pos="{-1.5 + 0.5*i:.2f} 0 1.0">\n'
            f'    <freejoint/>\n'
            f'    <geom name="g{i}" type="mesh" mesh="m{i}" rgba="{0.25+0.12*i:.2f} 0.55 0.85 1" '
            f'density="300" friction="1 0.1 0.01"/>\n'
            f'  </body>'
        )
    body_xml = "\n".join(bodies)
    return f"""<mujoco model="lack_table">
  <compiler angle="radian" meshdir="meshes" autolimits="true"/>
  <option gravity="0 0 -9.81" timestep="0.005"/>
  <asset>
{meshes}
    <texture name="grid" type="2d" builtin="checker" rgb1=".8 .8 .8" rgb2=".6 .6 .6" width="512" height="512"/>
    <material name="grid" texture="grid" texrepeat="5 5"/>
  </asset>
  <worldbody>
    <light pos="0 0 3" dir="0 0 -1"/>
    <geom name="floor" type="plane" size="3 3 0.1" material="grid"/>
    <camera name="overhead" pos="0 -2.2 2.0" xyaxes="1 0 0 0 0.6 0.8"/>
{body_xml}
  </worldbody>
</mujoco>"""

xml = build_scene_xml()
(SCENE_DIR / "scene.xml").write_text(xml)

model = mujoco.MjModel.from_xml_path(str(SCENE_DIR / "scene.xml"))
data  = mujoco.MjData(model)
mujoco.mj_forward(model, data)
print(f"model loaded: {model.nbody-1} bodies (exc. world), qpos dim = {model.nq}")

renderer = mujoco.Renderer(model, height=480, width=640)
renderer.update_scene(data, camera="overhead")
plt.figure(figsize=(6, 4)); plt.imshow(renderer.render()); plt.axis("off")
plt.title("scene — parts at their default positions"); plt.show()

## 3. Initial-state sampler and a `drop` primitive

- **`sample_init`**: each part gets a uniform xy in `[-0.8, 0.8]²` and a uniform SO(3) rotation
  via Shoemake 1992, then physics settles for 400 steps.
- **`drop(i, x, y)`**: teleport part `i` above `(x, y)`, zero its velocity, let physics settle.

In [ ]:
def shoemake_quat(rng: np.random.Generator) -> np.ndarray:
    u1, u2, u3 = rng.random(3)
    s1, s2 = np.sqrt(1 - u1), np.sqrt(u1)
    # MuJoCo quaternion order is (w, x, y, z)
    return np.array([
        s1 * np.cos(2 * np.pi * u2),
        s1 * np.sin(2 * np.pi * u2),
        s2 * np.sin(2 * np.pi * u3),
        s2 * np.cos(2 * np.pi * u3),
    ])

def set_free_body(model, data, name, pos, quat):
    bid  = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_BODY, name)
    jadr = model.body_jntadr[bid]
    qadr = model.jnt_qposadr[jadr]
    dadr = model.jnt_dofadr[jadr]
    data.qpos[qadr:qadr + 3]     = pos
    data.qpos[qadr + 3:qadr + 7] = quat
    data.qvel[dadr:dadr + 6]     = 0.0

def settle(model, data, steps=400):
    for _ in range(steps):
        mujoco.mj_step(model, data)

def sample_init(model, data, rng):
    for k, i in enumerate(PART_IDS):
        xy = rng.uniform(-0.8, 0.8, size=2)
        set_free_body(model, data, f"part_{i}", [xy[0], xy[1], 1.0 + 0.15 * k], shoemake_quat(rng))
    settle(model, data)

def drop(model, data, i, x, y, rng):
    set_free_body(model, data, f"part_{i}", [x, y, 1.1], shoemake_quat(rng))
    settle(model, data)

rng = np.random.default_rng(0)
sample_init(model, data, rng)
renderer.update_scene(data, camera="overhead")
plt.figure(figsize=(6, 4)); plt.imshow(renderer.render()); plt.axis("off")
plt.title("random init, settled under gravity"); plt.show()

## 4. Generate a transition corpus

`N = 16` transitions. For each: randomly init → snapshot `before` → pick leg `i ∈ {2,3,4,5}`
and drop it near a tabletop corner (with jitter) → snapshot `after`. A **state** is
`{part_idx: {"pos": np.ndarray(3), "quat": np.ndarray(4)}}`.

In [ ]:
def snapshot(model, data):
    out = {}
    for i in PART_IDS:
        bid = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_BODY, f"part_{i}")
        out[i] = {"pos": data.xpos[bid].copy(), "quat": data.xquat[bid].copy()}
    return out

def render_overhead():
    renderer.update_scene(data, camera="overhead")
    return renderer.render()

N = 16
rng = np.random.default_rng(42)
CORNERS = [(0.70, 0.40), (-0.70, 0.40), (0.70, -0.40), (-0.70, -0.40)]

transitions = []
img_before, img_after = [], []
for n in range(N):
    sample_init(model, data, rng)
    before = snapshot(model, data); ib = render_overhead()
    i = int(rng.choice([2, 3, 4, 5]))
    cx, cy = CORNERS[int(rng.integers(0, 4))]
    x = cx + float(rng.normal(0, 0.06)); y = cy + float(rng.normal(0, 0.06))
    drop(model, data, i, x, y, rng)
    after = snapshot(model, data); ia = render_overhead()
    transitions.append({"before": before, "after": after, "action": ("drop", i, float(x), float(y))})
    img_before.append(ib); img_after.append(ia)

print(f"collected {len(transitions)} transitions")
fig, ax = plt.subplots(2, 4, figsize=(14, 6))
for k in range(4):
    ax[0, k].imshow(img_before[k]); ax[0, k].axis("off"); ax[0, k].set_title(f"#{k} before")
    ax[1, k].imshow(img_after[k]);  ax[1, k].axis("off")
    ax[1, k].set_title(f"#{k} after  {transitions[k]['action']}")
plt.tight_layout(); plt.show()

## 5. VLM predicate imagination — Qwen2.5-VL-7B-Instruct (4-bit)

The VLM receives the manual step illustration + one before/after pair and proposes
boolean predicates as short Python expressions over the `state` dict. No seed vocabulary
is provided: the only primitives the VLM can use are `np.*` math on the raw pose fields.

In [ ]:
import torch
from transformers import AutoProcessor, AutoModelForImageTextToText, BitsAndBytesConfig

MODEL_ID = "Qwen/Qwen2.5-VL-7B-Instruct"
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16, bnb_4bit_quant_type="nf4")
processor = AutoProcessor.from_pretrained(MODEL_ID)
vlm = AutoModelForImageTextToText.from_pretrained(MODEL_ID, quantization_config=bnb, device_map="auto")
vlm.eval()
print("loaded:", MODEL_ID)

In [ ]:
import json, re
from PIL import Image as PILImage

SYSTEM = (
    "You are a reasoning assistant for robotic assembly. You are shown "
    "(1) an IKEA manual step illustration and (2) a 'before' and 'after' render of "
    "a physics simulation of one assembly action. Propose 3-6 boolean PREDICATES that "
    "describe state facts relevant to the manual step.\n"
    "Each predicate is a pure Python expression over a state dict `s` where "
    "`s[i]['pos']` is a length-3 numpy array (x,y,z) and `s[i]['quat']` is a "
    "length-4 MuJoCo quaternion (w,x,y,z). Part indices are 0..5. Part 0 is a "
    "large flat slab (the tabletop); parts 2,3,4,5 are four identical thin rods "
    "(the legs). Part 1 is a thin panel. You may only use `np.*` math on the pose "
    "fields — no other helpers.\n"
    "Return JSON ONLY. Schema: "
    '{"predicates": [{"name": "snake_case_name", "args": [int,...], '
    '"expr": "<python bool expression over s and args>"}]}. '
    "In `expr`, use `args[0]`, `args[1]`, … to index parts. "
    "Example: "
    '{"name": "leg_above_top", "args": [2, 0], '
    '"expr": "s[args[0]][\'pos\'][2] > s[args[1]][\'pos\'][2]"}.'
)

def _img_msg(img):
    return {"type": "image", "image": img if isinstance(img, PILImage.Image) else PILImage.fromarray(img)}

def call_vlm(manual, before, after, max_new_tokens=512):
    msgs = [
        {"role": "system", "content": SYSTEM},
        {"role": "user", "content": [
            {"type": "text",  "text": "Manual step illustration:"},
            _img_msg(manual),
            {"type": "text",  "text": "Simulation before:"},
            _img_msg(before),
            {"type": "text",  "text": "Simulation after:"},
            _img_msg(after),
            {"type": "text",  "text": "Propose predicates. Return JSON only."},
        ]},
    ]
    text = processor.apply_chat_template(msgs, add_generation_prompt=True, tokenize=False)
    imgs = [c["image"] for m in msgs for c in (m["content"] if isinstance(m["content"], list) else []) if c.get("type") == "image"]
    inputs = processor(text=[text], images=imgs, return_tensors="pt").to(vlm.device)
    with torch.no_grad():
        out = vlm.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    reply = processor.batch_decode(out[:, inputs.input_ids.shape[1]:], skip_special_tokens=True)[0]
    m = re.search(r"\{.*\}", reply, re.S)
    if not m:
        raise RuntimeError(f"VLM returned non-JSON output:\n{reply}")
    return json.loads(m.group(0))

proposal = call_vlm(manual_img, img_before[0], img_after[0])
print(json.dumps(proposal, indent=2))

## 6. Sim-verify predicates and induce the `drop` schema

For each proposed predicate we:
1. **Compile** the `expr` into a pure function `state -> bool` with only `np` exposed.
2. **Evaluate** it on the before and after states of all 16 transitions.
3. **Reject** if it raises, is trivially True everywhere, or trivially False everywhere.

From the surviving predicates we read off the STRIPS effects:
- `add` = predicates that flip `False → True` on **every** transition.
- `delete` = predicates that flip `True → False` on **every** transition.
- `pre` = predicates that are True on **every** before-state (always-held prerequisites).

In [ ]:
def compile_predicate(pred):
    code_obj = compile(pred["expr"], f"<pred_{pred['name']}>", "eval")
    args_tup = tuple(pred["args"])
    def fn(state):
        ctx = {"s": state, "args": args_tup, "np": np}
        for k, v in zip(["i", "j", "k", "l"], args_tup):
            ctx[k] = v
        return bool(eval(code_obj, {"__builtins__": {}, "np": np}, ctx))
    return fn

def eval_over_corpus(fn, transitions):
    bef = np.array([fn(t["before"]) for t in transitions])
    aft = np.array([fn(t["after"])  for t in transitions])
    return bef, aft

verified = []
for pred in proposal["predicates"]:
    try:
        fn = compile_predicate(pred)
        bef, aft = eval_over_corpus(fn, transitions)
    except Exception as e:
        print(f"REJECT {pred['name']}: {type(e).__name__}: {e}"); continue
    if bef.all() and aft.all():
        print(f"REJECT {pred['name']}: trivially True"); continue
    if (~bef).all() and (~aft).all():
        print(f"REJECT {pred['name']}: trivially False"); continue
    verified.append({"pred": pred, "fn": fn, "before": bef, "after": aft})
    print(f"OK     {pred['name']}{tuple(pred['args'])}: before={bef.sum()}/{len(bef)} after={aft.sum()}/{len(aft)}")

add_set, del_set, pre_set = [], [], []
for v in verified:
    name_sig = f"{v['pred']['name']}{tuple(v['pred']['args'])}"
    becomes_true  = (~v["before"]) & v["after"]
    becomes_false = v["before"] & (~v["after"])
    if becomes_true.all()  and becomes_true.any():  add_set.append(name_sig)
    if becomes_false.all() and becomes_false.any(): del_set.append(name_sig)
    if v["before"].all(): pre_set.append(name_sig)

print("\n=== Induced STRIPS schema for action 'drop(i, x, y)' ===")
print(f"preconditions : {pre_set}")
print(f"add effects   : {add_set}")
print(f"delete effects: {del_set}")

## Summary

End-to-end, in one notebook: real LACK meshes → MuJoCo physics → VLM reasons about the
manual plus before/after renders → proposes predicates as Python expressions → notebook
sim-verifies them and induces the `drop` action's STRIPS schema.

**Extending.** (i) Add more primitive actions (e.g., `align_site(i, j)`, `push(i, dx, dy)`)
and cluster transitions by action type before inducing schemas. (ii) Include more manual
steps by indexing `step_*_no_seg.png`. (iii) Swap in Gemini 2.5 Pro by replacing the
`call_vlm` body with a `google-genai` call if VPN access permits.